# 09 — Guardrails and human approval
Validate before the Crew runs. Check the report before accepting it. A mock
email is a side effect, so it requires approval immediately before execution.



**Basic lesson 9 of 13.**


## Problem

An agent's answer must not authorize its own customer-facing side effect.


## Concept

Validation and human approval are executable boundaries outside the model.

Guardrails are executable checks at input, output, and action boundaries. A human approval decision is external state; no prompt can manufacture it.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call   ◀ YOU ARE HERE
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email   ◀ YOU ARE HERE

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
input ──▶ validate_question ──▶ Crew report
                                      │
                               validate_report
                                      │
                           approval? ──no──▶ blocked
                               │ yes
                               ▼
                         idempotent mock email
```

request → validation → report check → approval gate → mock action. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Reject an invalid question and validate a deliberately sourced report

Reject an invalid question and validate a deliberately sourced report.


In [ ]:
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.control import mock_email, validate_question, validate_report
from workshop.schemas import DecisionReport, Evidence

try:
    validate_question("hi")
except ValueError as error:
    print("Input blocked:", error)

report = DecisionReport(
    recommendation="pilot",
    rationale="A private deployment is possible, but deletion and throughput need verification.",
    evidence=[
        Evidence(source_id="architecture.md", claim="Private VPC deployment is possible."),
        Evidence(source_id="security.md", claim="Data deletion guarantees are unverified."),
    ],
    risks=["Unverified deletion guarantees"],
    next_steps=["Run a bounded pilot"],
)
print("Output check:", validate_report(report))



## Exercise the approval boundary
First run is denied. Second run records a local mock action. A repeated
action ID is ignored, which matters when an interrupted Flow resumes.



## Step 2 — Exercise denied, approved, and replayed mock actions using the same ID

Exercise denied, approved, and replayed mock actions using the same ID.


In [ ]:
print(mock_email(report, approved=False, action_id="lesson-09-demo"))
print(mock_email(report, approved=True, action_id="lesson-09-demo"))
print(mock_email(report, approved=True, action_id="lesson-09-demo"))



## Bonus scenario — an email auto-responder
A task guardrail checks a draft before it is accepted (inspired by CrewAI's
*Email Auto Responder* flow). Test the check on plain text first.



In [ ]:
from crewai import Agent, Crew, Process, Task, TaskOutput

from workshop.common import model


def no_promises(output: TaskOutput) -> tuple[bool, str]:
    if "guarantee" in output.raw.lower():
        return False, "Do not promise guarantees; offer to look into it instead."
    return True, output.raw


print(no_promises(TaskOutput(description="d", raw="We guarantee a refund.", agent="a")))
print(no_promises(TaskOutput(description="d", raw="We will look into your order.", agent="a")))

support = Agent(role="Support Agent", goal="Reply politely", backstory="You never over-promise.", llm=model(), allow_delegation=False)
reply = Task(
    description="Reply in two sentences to a customer whose parcel is late.",
    expected_output="A polite two-sentence reply.",
    agent=support,
    guardrail=no_promises,
    guardrail_max_retries=1,
)
print(Crew(agents=[support], tasks=[reply], process=Process.sequential).kickoff().raw)


## Failure mode to know about

Never treat an agent's request as human approval. An action must pass validation and an external approval flag; retries must reuse an idempotency key.


## Production implication

Authorize side effects through the host application, store approval evidence, and use idempotency keys. Model output is never an authorization token.


## Your turn

Run the denied, approved, and repeated-action cases in order.

**Checkpoint:** Denied writes nothing; approved writes one local record; repeat writes no second record.
